In [ ]:
%pip install accelerate --quiet
print("✓ accelerate installed")

In [ ]:
# ## 2 · Imports
 
# %%
import os, json, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2BertModel,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
 
warnings.filterwarnings("ignore")
print("✓ All imports successful")

In [ ]:
# ## 3 · Device & reproducibility
 
# %%
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu  = torch.cuda.device_count()
print(f"Device : {device}  |  GPUs available: {n_gpu}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

In [ ]:
# %%
CONFIG = {
    # ── model ─────────────────────────────────────────────────────────────
    "model_id"   : "S3",
    "model_name" : "facebook/w2v-bert-2.0",
    "num_labels" : 7,
 
    # ── training ──────────────────────────────────────────────────────────
    "epochs"                     : 20,
    "early_stopping_patience"    : 4, 
    "batch_size"                 : 8,       # per GPU; effective = batch × accum
    "gradient_accumulation_steps": 4,       # effective batch = 32
    "lr_transformer"             : 2e-5,
    "lr_head"                    : 1e-4,
    "weight_decay"               : 0.01,
    "warmup_ratio"               : 0.1,
    "max_audio_len_sec"          : 6.0,     # clips longer than this are truncated
    "sample_rate"                : 16_000,
 
    # ── mixed precision & memory ──────────────────────────────────────────
    "fp16"                       : torch.cuda.is_available(),
    "gradient_checkpointing"     : True,    # saves ~30 % VRAM
 
    # ── Kaggle paths ──────────────────────────────────────────────────────
    # Dataset slugs as added in Notebook settings → Add data
    "ravdess_dir" : "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio",
    "tess_dir"    : "/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess",
    "output_dir"  : "/kaggle/working/S3_wav2vec_bert_ser/",
}
 
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("CONFIG loaded  |  output_dir →", CONFIG["output_dir"])
 

In [ ]:
# ## 5 · Emotion label mapping
#
# RAVDESS uses numeric codes; TESS encodes emotion in the filename suffix.
# We map both to a shared 7-class scheme (disgust is absent in TESS, so it
# will only come from RAVDESS).
 
# %%
EMOTION_MAP = {
    "neutral" : 0,
    "calm"    : 1,
    "happy"   : 2,
    "sad"     : 3,
    "angry"   : 4,
    "fearful" : 5,
    "disgust" : 6,
}
IDX2LABEL = {v: k for k, v in EMOTION_MAP.items()}
 
# RAVDESS: emotion code is the 3rd segment of the filename (1-indexed)
RAVDESS_CODE = {
    "01": "neutral", "02": "calm",    "03": "happy",
    "04": "sad",     "05": "angry",   "06": "fearful",
    "07": "disgust", "08": None,      # surprised → skip (not in our 7-class)
}
 
# TESS: last word before .wav, lower-cased
TESS_MAP = {
    "neutral": "neutral", "happy": "happy",  "sad": "sad",
    "angry"  : "angry",   "fear" : "fearful","disgust": "disgust",
    "ps"     : None,      # pleasant surprised → skip
}

In [ ]:
# ## 6 · Build file list
 
# %%
def collect_ravdess(root):
    rows = []
    for actor_dir in sorted(os.listdir(root)):
        actor_path = os.path.join(root, actor_dir)
        if not os.path.isdir(actor_path):
            continue
        for fname in os.listdir(actor_path):
            if not fname.endswith(".wav"):
                continue
            parts = fname.replace(".wav", "").split("-")
            if len(parts) < 7:
                continue
            emotion_code = parts[2]
            label_str = RAVDESS_CODE.get(emotion_code)
            if label_str is None:
                continue
            rows.append({
                "path"   : os.path.join(actor_path, fname),
                "label"  : EMOTION_MAP[label_str],
                "source" : "RAVDESS",
            })
    return rows
 
 
def collect_tess(root):
    rows = []
    for sub in sorted(os.listdir(root)):
        sub_path = os.path.join(root, sub)
        if not os.path.isdir(sub_path):
            continue
        for fname in os.listdir(sub_path):
            if not fname.lower().endswith(".wav"):
                continue
            emotion_str = fname.lower().replace(".wav", "").split("_")[-1]
            label_str   = TESS_MAP.get(emotion_str)
            if label_str is None:
                continue
            rows.append({
                "path"   : os.path.join(sub_path, fname),
                "label"  : EMOTION_MAP[label_str],
                "source" : "TESS",
            })
    return rows
 
 
rows  = collect_ravdess(CONFIG["ravdess_dir"])
rows += collect_tess(CONFIG["tess_dir"])
df    = pd.DataFrame(rows)
 
print(f"Total samples : {len(df)}")
print(df["label"].map(IDX2LABEL).value_counts())
 

In [ ]:
# ## 7 · Train / val / test split (stratified)
 
# %%
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)
print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")
 
# %% [markdown]

In [ ]:
# ## 8 · Feature extractor
 
# %%
feature_extractor = AutoFeatureExtractor.from_pretrained(CONFIG["model_name"])
print(f"✓ Feature extractor loaded  (sampling_rate={feature_extractor.sampling_rate})")

In [ ]:
# ## 9 · Dataset class
 
# %%
MAX_LEN = int(CONFIG["max_audio_len_sec"] * CONFIG["sample_rate"])
 
class SERDataset(Dataset):
    def __init__(self, dataframe, feature_extractor, max_len=MAX_LEN):
        self.df  = dataframe.reset_index(drop=True)
        self.fe  = feature_extractor
        self.max = max_len
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = librosa.load(row["path"], sr=CONFIG["sample_rate"], mono=True)
 
        # truncate / zero-pad
        if len(audio) > self.max:
            audio = audio[:self.max]
        else:
            audio = np.pad(audio, (0, self.max - len(audio)))
 
        inputs = self.fe(
            audio,
            sampling_rate=CONFIG["sample_rate"],
            return_tensors="pt",
            padding=False,
        )
        return {
            "input_features": inputs["input_features"].squeeze(0),
            "label"         : torch.tensor(row["label"], dtype=torch.long),
        }
 
 
def collate_fn(batch):
    input_features = torch.stack([b["input_features"] for b in batch])
    labels         = torch.stack([b["label"]          for b in batch])
    return {"input_features": input_features, "labels": labels}
 
 
train_ds = SERDataset(train_df, feature_extractor)
val_ds   = SERDataset(val_df,   feature_extractor)
test_ds  = SERDataset(test_df,  feature_extractor)
 
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                          shuffle=True,  collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"] * 2,
                          shuffle=False, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"] * 2,
                          shuffle=False, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)
 
print(f"✓ DataLoaders ready  —  steps/epoch (train): {len(train_loader)}")
 

In [ ]:
class ScalarMixing(nn.Module):
    """Learns a softmax-normalised weighted sum over all hidden states."""
    def __init__(self, num_layers: int = 25):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(num_layers))

    def forward(self, hidden_states):                  # tuple of (B, T, D)
        w = F.softmax(self.weights, dim=0)             # (num_layers,)
        stacked = torch.stack(hidden_states, dim=0)    # (L, B, T, D)
        return (w[:, None, None, None] * stacked).sum(dim=0)  # (B, T, D)


class W2VBertSER(nn.Module):
    def __init__(self, model_name: str, num_labels: int):
        super().__init__()
        self.encoder = Wav2Vec2BertModel.from_pretrained(
            model_name,
            output_hidden_states=True,
        )
        if CONFIG["gradient_checkpointing"]:
            self.encoder.gradient_checkpointing_enable()

        hidden_dim = self.encoder.config.hidden_size   # 1024 for w2v-bert-2.0
        self.mixing = ScalarMixing(num_layers=25)      # embedding + 24 layers

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_features, labels=None):
        outputs = self.encoder(input_features)
        mixed   = self.mixing(outputs.hidden_states)   # (B, T, D)
        pooled  = mixed.mean(dim=1)                    # (B, D)
        logits  = self.classifier(pooled)              # (B, C)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return {"loss": loss, "logits": logits}


model = W2VBertSER(CONFIG["model_name"], CONFIG["num_labels"])

# Multi-GPU support (Kaggle T4 x2)
if n_gpu > 1:
    model = nn.DataParallel(model)

model = model.to(device)
print(f"✓ Model on {device}  |  DataParallel: {n_gpu > 1}")

# ── safety checkpoint: write initial weights so disk always has something ──
_init_path = os.path.join(CONFIG["output_dir"], "best_model.pt")
_init_state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
torch.save(_init_state, _init_path)
print(f"✓ Initial checkpoint written → {_init_path}  "
      f"({os.path.getsize(_init_path)/1024**2:.1f} MB)")

In [ ]:
# ## 11 · Optimizer & scheduler
 
# %%
# Separate learning rates: lower for pretrained encoder, higher for new head
def get_param_groups(model):
    encoder = model.module.encoder if hasattr(model, "module") else model.encoder
    mixing  = model.module.mixing  if hasattr(model, "module") else model.mixing
    clf     = model.module.classifier if hasattr(model, "module") else model.classifier
 
    encoder_params = list(encoder.parameters())
    head_params    = list(mixing.parameters()) + list(clf.parameters())
    return [
        {"params": encoder_params, "lr": CONFIG["lr_transformer"]},
        {"params": head_params,    "lr": CONFIG["lr_head"]},
    ]
 
 
optimizer = torch.optim.AdamW(
    get_param_groups(model),
    weight_decay=CONFIG["weight_decay"],
)
 
total_steps  = (len(train_loader) // CONFIG["gradient_accumulation_steps"]) * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = GradScaler(enabled=CONFIG["fp16"])
 
print(f"Total optimiser steps: {total_steps}  |  Warmup: {warmup_steps}")
 

In [ ]:
# ## 12 · Training loop
 
# %%
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    grad_step = 0
 
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, desc="Train" if training else "Val ", leave=False)
        for step, batch in enumerate(pbar):
            input_features = batch["input_features"].to(device)
            labels         = batch["labels"].to(device)
 
            with autocast(enabled=CONFIG["fp16"]):
                out  = model(input_features=input_features, labels=labels)
                loss = out["loss"]
                if n_gpu > 1:
                    loss = loss.mean()
                loss = loss / CONFIG["gradient_accumulation_steps"]
 
            if training:
                scaler.scale(loss).backward()
                if (step + 1) % CONFIG["gradient_accumulation_steps"] == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()
                    grad_step += 1
 
            total_loss += loss.item() * CONFIG["gradient_accumulation_steps"]
            preds = out["logits"].argmax(dim=-1).cpu().tolist()
            all_preds  += preds
            all_labels += labels.cpu().tolist()
 
            pbar.set_postfix(loss=f"{loss.item() * CONFIG['gradient_accumulation_steps']:.4f}")
 
    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, f1
 
 
history    = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
              "train_f1" : [], "val_f1" : []}
best_val_f1     = 0.0
best_model_path = os.path.join(CONFIG["output_dir"], "best_model.pt")
 
patience_counter = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{CONFIG['epochs']}")

    t_loss, t_acc, t_f1 = run_epoch(train_loader, training=True)
    v_loss, v_acc, v_f1 = run_epoch(val_loader,   training=False)

    history["train_loss"].append(t_loss); history["val_loss"].append(v_loss)
    history["train_acc" ].append(t_acc);  history["val_acc" ].append(v_acc)
    history["train_f1"  ].append(t_f1);  history["val_f1"  ].append(v_f1)

    print(f"  Train  — loss: {t_loss:.4f}  acc: {t_acc:.4f}  F1: {t_f1:.4f}")
    print(f"  Val    — loss: {v_loss:.4f}  acc: {v_acc:.4f}  F1: {v_f1:.4f}")

    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
        torch.save(state, best_model_path)
        patience_counter = 0
        print(f"  ✓ Best model saved  (val F1 = {best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{CONFIG['early_stopping_patience']})")
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print("  Early stopping triggered.")
            break

print("\n✓ Training complete")
 

In [ ]:
# ── force clear everything still holding VRAM ─────────────────────────────
for var in ['model', 'eval_model', 'core_model', 'optimizer', 'scheduler', 'scaler']:
    if var in globals():
        del globals()[var]

torch.cuda.empty_cache()
import gc; gc.collect()
torch.cuda.empty_cache()  # call twice — first gc.collect frees python refs, second flush clears them
print(f"VRAM free after cleanup: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")

# ── load checkpoint to CPU first, then move to GPU ───────────────────────
best_state = torch.load(best_model_path, map_location="cpu")  # ← cpu, not device
core_model = W2VBertSER(CONFIG["model_name"], CONFIG["num_labels"])
core_model.load_state_dict(best_state)
core_model = core_model.to(device)  # move after loading
core_model.eval()

# ── rebuild test loader with small batch ──────────────────────────────────
test_loader = DataLoader(test_ds, batch_size=4,
                         shuffle=False, collate_fn=collate_fn,
                         num_workers=2, pin_memory=True)

# ── evaluate ──────────────────────────────────────────────────────────────
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test"):
        input_features = batch["input_features"].to(device)
        labels         = batch["labels"].to(device)
        with autocast(enabled=CONFIG["fp16"]):
            out = core_model(input_features=input_features)
        preds = out["logits"].argmax(dim=-1).cpu().tolist()
        all_preds  += preds
        all_labels += labels.cpu().tolist()

test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
print(f"\nTest Accuracy     : {test_acc:.4f}")
print(f"Test F1 (weighted): {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds,
                            target_names=[IDX2LABEL[i] for i in range(CONFIG["num_labels"])],
                            zero_division=0))

In [ ]:
# ## 14 · Plots
 
# %%
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
for ax, metric, title in zip(
    axes,
    [("train_loss","val_loss"), ("train_acc","val_acc"), ("train_f1","val_f1")],
    ["Loss", "Accuracy", "F1 (weighted)"],
):
    ax.plot(history[metric[0]], label="Train")
    ax.plot(history[metric[1]], label="Val")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend()
 
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "training_curves.png"), dpi=150)
plt.show()
 
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[IDX2LABEL[i] for i in range(CONFIG["num_labels"])],
            yticklabels=[IDX2LABEL[i] for i in range(CONFIG["num_labels"])], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — S3 (Test set)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
# ## 15 · Save summary JSON
 
# %%
summary = {
    "model_id"   : CONFIG["model_id"],
    "model_name" : CONFIG["model_name"],
    "dataset"    : "RAVDESS + TESS (unified, 7-class)",
    "num_labels" : CONFIG["num_labels"],
    "test_accuracy"   : round(test_acc, 4),
    "test_f1_weighted": round(test_f1, 4),
    "best_val_f1"     : round(best_val_f1, 4),
    "epochs"          : CONFIG["epochs"],
    "fp16"            : CONFIG["fp16"],
    "gradient_checkpointing": CONFIG["gradient_checkpointing"],
    "history"    : history,
}
with open(os.path.join(CONFIG["output_dir"], "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
 
print(f"✓ Summary saved → {CONFIG['output_dir']}summary.json")
print(f"\n{'='*60}")
print(f"S3 complete  |  Test Acc: {test_acc:.4f}  |  Test F1: {test_f1:.4f}")
print(f"Artefacts in: {CONFIG['output_dir']}")

In [ ]:
# see what's still in memory
import torch
print([k for k in globals().keys() if not k.startswith('_')])